In [42]:
import pandas as pd
from pathlib import Path

In [43]:
PROJECT_ROOT = Path.cwd().parent
gangnam_df = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "gangnam" / "gangnam_2016_2025.csv", low_memory=False)
seocho_df = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "seocho" / "seocho_2016_2025.csv", low_memory=False)

In [44]:
df = pd.concat([gangnam_df, seocho_df], ignore_index=True)
df["dealAmount"] = (
    df["dealAmount"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.strip()
    .astype(int)
)
df["is_cancelled"] = df["cdealType"].astype(str).str.strip() == "O"
df_clean = df[~df["is_cancelled"]].copy()

df_clean["age"] = df_clean["dealYear"] - df_clean["buildYear"]
df_clean["price_per_area"] = df_clean["dealAmount"] / df_clean["excluUseAr"]
df_clean["is_presale"] = df_clean["age"] < 0

In [45]:
bins = [0, 40, 60, 85, 135, 300]
labels = ["소형", "국민평형", "중형", "대형", "초대형"]
df_clean["area_group"] = pd.cut(df_clean["excluUseAr"], bins=bins, labels=labels)

df_clean["deal_month"] = df_clean["dealMonth"]
df_clean["deal_year"] = df_clean["dealYear"]
df_clean["is_reconstruction_candidate"] = (df_clean["age"] >= 30).astype(int)

feature_cols = ["age", "floor", "excluUseAr", "is_presale", "is_reconstruction_candidate",
                 "area_group", "deal_month", "deal_year", "gu_name", "umdNm", "aptNm"]
target_col = "price_per_area"

model_df = df_clean[feature_cols + [target_col]].copy()
model_df.shape

id_cols = df_clean.loc[model_df.index, ["aptNm", "dealYear", "dealMonth"]]

In [46]:
train_mask = model_df["deal_year"] <= 2023

apt_mean_price = model_df.loc[train_mask].groupby("aptNm")["price_per_area"].mean()
global_mean_price = model_df.loc[train_mask, "price_per_area"].mean()

model_df["aptNm_encoded"] = model_df["aptNm"].map(apt_mean_price).fillna(global_mean_price)
model_df = model_df.drop(columns=["aptNm"])

model_df = pd.get_dummies(model_df, columns=["area_group", "gu_name", "umdNm"], drop_first=True)
model_df["is_presale"] = model_df["is_presale"].astype(int)
model_df = model_df.dropna()
model_df.shape

(67743, 35)

In [47]:
train_df = model_df[model_df["deal_year"] <= 2023]
test_df = model_df[model_df["deal_year"] >= 2024]

print(f"Train: {len(train_df):,}건 (2016~2023)")
print(f"Test: {len(test_df):,}건 (2024~2025)")

Train: 54,322건 (2016~2023)
Test: 13,421건 (2024~2025)


In [48]:
save_path = PROJECT_ROOT / "data" / "processed"
save_path.mkdir(parents=True, exist_ok=True)
train_df.to_csv(save_path / "train.csv", index=False)
test_df.to_csv(save_path / "test.csv", index=False)